In [1]:
import pandas as pd  # Import pandas for structured data analysis.
import numpy as np  # Import NumPy for numerical diagnostics.
from pathlib import Path  # Import Path for file-safe references.
from IPython.display import display  # Import display for readable audit tables.
from scipy import stats  # Import statistical tests for audit checks.
FILE_PATH = "raw_data/customers.csv"  # Point to the project raw dataset.
audit_findings = []  # Create a register for evidence-based findings.
print("BUSINESS INSIGHT: Customer Retention and Churn")  # State the consulting context for the audit.
print("BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.")  # State the business problem being investigated.
print("AUDIT LENS: retention, repeat purchase, payment friction, service experience")  # State the signals relevant to this problem.


BUSINESS INSIGHT: Customer Retention and Churn
BUSINESS PROBLEM: Identify customer behaviours associated with inactivity and churn.
AUDIT LENS: retention, repeat purchase, payment friction, service experience


In [4]:
data_path = Path(FILE_PATH)
if not data_path.is_absolute():
    candidates = [Path.cwd() / data_path] + [parent / data_path for parent in Path.cwd().parents]
    data_path = next((path for path in candidates if path.is_file()), data_path)

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {FILE_PATH}")

df = pd.read_csv(data_path)  # Load the raw dataset.
print(f"Loaded {data_path.name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
display(df.head())# Inspect representative raw records.
print(f"Loaded {Path(FILE_PATH).name}")  # Confirm the source dataset loaded.
print(f"Rows: {len(df):,}")  # Show the available observation count.
print(f"Columns: {len(df.columns):,}")  # Show the available field count.
display(df.head())  # Inspect representative raw records.


Loaded customers.csv
Rows: 12,000
Columns: 15


,customer_id,legacy_customer_id,first_name,last_name,date_of_birth,gender,email,phone,registration_date,acquisition_channel,home_city,province,customer_segment,loyalty_status,customer_status
0,CUS-000001,1.0,Kagiso,Mthembu,1964-11-23,Female,kagiso.mthembu1@example.com,27806794542,2020-12-09,Store,Pretoria,Gauteng,High Value,Bronze,Active
1,CUS-000002,2.0,Zanele,Jacobs,1968-12-24,Male,zanele.jacobs2@example.com,27808213503,2022-12-23,Referral,Umhlanga,KwaZulu-Natal,Emerging,Silver,Active
2,CUS-000003,3.0,Naledi,Botha,1996-07-05,Prefer not to say,NaN,27860397578,2019-02-23,Store,Bloemfontein,Free State,Core,Bronze,Active
3,CUS-000004,4.0,Aisha,Botha,1984-08-31,Male,NaN,27648953446,2018-02-07,Website,Umhlanga,KwaZulu-Natal,Standard,Silver,Inactive
4,CUS-000005,NaN,Michael,Mokoena,2002-06-24,Male,michael.mokoena5@example.com,27689135499,2025-10-08,Store,Gqeberha,Eastern Cape,Standard,Bronze,Active


Loaded customers.csv
Rows: 12,000
Columns: 15


,customer_id,legacy_customer_id,first_name,last_name,date_of_birth,gender,email,phone,registration_date,acquisition_channel,home_city,province,customer_segment,loyalty_status,customer_status
0,CUS-000001,1.0,Kagiso,Mthembu,1964-11-23,Female,kagiso.mthembu1@example.com,27806794542,2020-12-09,Store,Pretoria,Gauteng,High Value,Bronze,Active
1,CUS-000002,2.0,Zanele,Jacobs,1968-12-24,Male,zanele.jacobs2@example.com,27808213503,2022-12-23,Referral,Umhlanga,KwaZulu-Natal,Emerging,Silver,Active
2,CUS-000003,3.0,Naledi,Botha,1996-07-05,Prefer not to say,NaN,27860397578,2019-02-23,Store,Bloemfontein,Free State,Core,Bronze,Active
3,CUS-000004,4.0,Aisha,Botha,1984-08-31,Male,NaN,27648953446,2018-02-07,Website,Umhlanga,KwaZulu-Natal,Standard,Silver,Inactive
4,CUS-000005,NaN,Michael,Mokoena,2002-06-24,Male,michael.mokoena5@example.com,27689135499,2025-10-08,Store,Gqeberha,Eastern Cape,Standard,Bronze,Active


In [5]:
overview = pd.DataFrame({"metric":["rows","columns","duplicates","missing_cells"],"value":[len(df),len(df.columns),int(df.duplicated().sum()),int(df.isna().sum().sum())]})  # Build an initial data-quality summary.
display(overview)  # Review the initial quality position.
print("Decision point: determine which findings require remediation.")  # Make the audit decision explicit.


,metric,value
0,rows,12000
1,columns,15
2,duplicates,0
3,missing_cells,11619


Decision point: determine which findings require remediation.


In [6]:
schema = pd.DataFrame({"column":df.columns,"dtype":df.dtypes.astype(str).values,"non_null":df.notna().sum().values,"missing":df.isna().sum().values,"unique":df.nunique(dropna=True).values})  # Profile schema completeness and cardinality.
display(schema)  # Inspect field-level structural evidence.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()  # Identify numeric fields for statistical checks.
text_cols = df.select_dtypes(include=["object","string"]).columns.tolist()  # Identify text fields for categorical checks.
date_cols = [c for c in df.columns if "date" in c.lower() or "time" in c.lower()]  # Identify likely temporal fields.


,column,dtype,non_null,missing,unique
0,customer_id,str,12000,0,12000
1,legacy_customer_id,float64,7067,4933,7067
2,first_name,str,12000,0,10
3,last_name,str,12000,0,10
4,date_of_birth,str,10193,1807,8184
5,gender,str,12000,0,3
6,email,str,10845,1155,10845
7,phone,str,11330,670,11329
8,registration_date,str,12000,0,3046
9,acquisition_channel,str,12000,0,5


In [7]:
missing = df.isna().sum().sort_values(ascending=False)  # Measure explicit missingness by field.
missing = missing[missing.gt(0)]  # Keep only fields with missing values.
display(missing.to_frame("missing_count"))  # Inspect missing-value concentration.
for col in missing.index: audit_findings.append({"issue":"missing_values","column":col,"count":int(missing[col])})  # Register observed missing-value evidence.


,missing_count
legacy_customer_id,4933
loyalty_status,3054
date_of_birth,1807
email,1155
phone,670


In [8]:
duplicates = int(df.duplicated().sum())  # Measure exact duplicate records.
audit_findings.append({"issue":"duplicate_rows","count":duplicates})  # Register duplicate-row evidence.
print(f"Duplicate rows identified: {duplicates:,}")  # Report duplicate-row evidence.


Duplicate rows identified: 0


In [9]:
numeric_audit = df[numeric_cols].describe().T if numeric_cols else pd.DataFrame()  # Summarise numeric distributions.
display(numeric_audit)  # Inspect scale, spread and potential extremes.
if numeric_cols: outlier_rates = ((df[numeric_cols] < df[numeric_cols].quantile(.25) - 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25))) | (df[numeric_cols] > df[numeric_cols].quantile(.75) + 1.5*(df[numeric_cols].quantile(.75)-df[numeric_cols].quantile(.25)))).mean().sort_values(ascending=False)  # Estimate IQR-based extreme-value rates.
if numeric_cols: display(outlier_rates.to_frame("iqr_extreme_rate"))  # Inspect fields requiring business review.


,count,mean,std,min,25%,50%,75%,max
legacy_customer_id,7067.0,5970.626291,3459.937801,1.0,2960.5,5983.0,8977.5,12000.0


,iqr_extreme_rate
legacy_customer_id,0.0


In [10]:
category_audit = []  # Create categorical consistency checks.
for col in text_cols: category_audit.append({"column":col,"unique":int(df[col].nunique(dropna=True)),"blank":int(df[col].astype("string").str.strip().eq("").sum()),"top_values":df[col].value_counts(dropna=False).head(5).to_dict()})  # Profile text fields for inconsistent values.
display(pd.DataFrame(category_audit))  # Review categorical concentration and blanks.


,column,unique,blank,top_values
0,customer_id,12000,0,"{'CUS-000001': 1, 'CUS-000002': 1, 'CUS-000003..."
1,first_name,10,0,"{'Nomsa': 1251, 'Aisha': 1220, 'Sipho': 1208, ..."
2,last_name,10,0,"{'Jacobs': 1247, 'Williams': 1234, 'Mthembu': ..."
3,date_of_birth,8184,0,"{nan: 1807, '1995-06-06': 5, '1949-03-05': 5, ..."
4,gender,3,0,"{'Prefer not to say': 4081, 'Male': 3977, 'Fem..."
5,email,10845,0,"{nan: 1155, 'kagiso.mthembu1@example.com': 1, ..."
6,phone,11329,0,"{nan: 670, '27814451235': 2, '27806794542': 1,..."
7,registration_date,3046,0,"{'2024-09-22': 13, '2026-03-09': 13, '2018-08-..."
8,acquisition_channel,5,0,"{'Store': 4080, 'Website': 3381, 'Mobile': 224..."
9,home_city,11,0,"{'Pretoria': 1145, 'Umhlanga': 1133, 'Durban':..."


In [11]:
date_audit = []  # Create temporal field diagnostics.
for col in date_cols: parsed = pd.to_datetime(df[col], errors="coerce"); date_audit.append({"column":col,"parse_failures":int(parsed.isna().sum()-df[col].isna().sum()),"min":parsed.min(),"max":parsed.max()})  # Test temporal fields for parseability and range.
display(pd.DataFrame(date_audit))  # Review date integrity before analysis.


,column,parse_failures,min,max
0,date_of_birth,0,1945-01-01,2005-03-26
1,registration_date,0,2018-01-01,2026-06-27


In [12]:
identifier_audit = []  # Create identifier uniqueness diagnostics.
for col in df.columns: identifier_audit.append({"column":col,"unique_ratio":round(df[col].nunique(dropna=True)/max(len(df),1),3)})  # Measure field-level uniqueness.
identifier_audit = pd.DataFrame(identifier_audit).sort_values("unique_ratio",ascending=False)  # Rank potential identifiers and keys.
display(identifier_audit.head(15))  # Inspect candidate identifiers and high-cardinality fields.


,column,unique_ratio
0,customer_id,1.000
7,phone,0.944
6,email,0.904
4,date_of_birth,0.682
1,legacy_customer_id,0.589
8,registration_date,0.254
2,first_name,0.001
3,last_name,0.001
10,home_city,0.001
11,province,0.001


In [13]:
numeric_pairs = []  # Create relationship diagnostics for numeric fields.
if len(numeric_cols) > 1: numeric_pairs = df[numeric_cols].corr(numeric_only=True).stack().reset_index(name="correlation")  # Measure numeric relationships for diagnostic context.
if numeric_pairs != []: display(numeric_pairs.sort_values("correlation",key=lambda s:s.abs(),ascending=False).head(20))  # Inspect strongest observed numeric relationships.


In [14]:
finding_table = pd.DataFrame(audit_findings)  # Convert findings into a reviewable audit register.
if finding_table.empty: finding_table = pd.DataFrame([{ "issue":"none_detected_by_template", "count":0 }])  # Record when automated checks find no issues.
display(finding_table)  # Review the evidence before remediation.
print("Consulting decision: validate material findings against business rules before cleaning.")  # Prevent automatic treatment of every anomaly as an error.


,issue,column,count
0,missing_values,legacy_customer_id,4933
1,missing_values,loyalty_status,3054
2,missing_values,date_of_birth,1807
3,missing_values,email,1155
4,missing_values,phone,670
5,duplicate_rows,NaN,0


Consulting decision: validate material findings against business rules before cleaning.


In [15]:
stem = Path(FILE_PATH).stem  # Capture the dataset stem for output naming.
audit_summary = pd.DataFrame({"dataset":[Path(FILE_PATH).name],"rows":[len(df)],"columns":[len(df.columns)],"duplicates":[duplicates],"missing_cells":[int(df.isna().sum().sum())]})  # Create an auditable executive summary.
display(audit_summary)  # Present the final audit snapshot.
audit_summary.to_csv(f"outputs/{stem}_audit_summary.csv",index=False)  # Save the audit summary for downstream review.


,dataset,rows,columns,duplicates,missing_cells
0,customers.csv,12000,15,0,11619


OSError: Cannot save file into a non-existent directory: 'outputs'